# 02 — Data Cleaning Pipeline

## Overview

This notebook implements the deterministic cleaning and integration pipeline for the audited higher education datasets.

The transformations performed at this stage are restricted to explicit and reproducible preprocessing decisions identified during the audit workflow, including:

- removal of redundant operational metadata,
- normalization of categorical scheduling variables,
- reconciliation of minor enrollment metric inconsistencies,
- and integration of operational and program-level reference data.

The resulting output is a consolidated analysis-ready dataset intended for downstream exploratory analysis and statistical evaluation.

In [1]:
from notebook_utils import ensure_repo_root

# Establish the repository root as the working directory for this notebook
ensure_repo_root()

WindowsPath('C:/Github/higher-education-outcomes-analysis')

## Load Audited Canonical Datasets

The cleaned canonical tables generated during the audit stage are loaded as the baseline input for deterministic preprocessing and integration.

All transformations performed in this notebook operate exclusively on sanitized intermediate datasets rather than raw institutional exports.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_utils import load_data, set_pandas_display_options

enrollment = load_data("enrollment")
programs = load_data("programs")
offering = load_data("offering")

# Set pandas display options for better readability
set_pandas_display_options()

In [3]:
from src.config.mappings import (
    CANONICAL_MAPPINGS,
    TYPO_MAPPINGS,
)

from src.config.columns import (
    ENROLLMENT_REDUNDANT_METADATA_COLS,
    ENROLLMENT_METRIC_COMPONENT_COLS,
    OFFERING_TEXT_COLS,
)

from src.cleaning import (
    drop_columns,
    normalize_text_columns,
    correct_typo_variants,
    apply_canonical_taxonomy,
    reconcile_metric_totals,
)

from src.utils.text import normalize_text

## Remove Redundant Operational Metadata

The enrollment table contains several sparse operational metadata fields related to scheduling and instructional modality.

These attributes are not removed solely due to missingness levels. Instead, they are excluded because the `Offering` dataset constitutes the canonical operational metadata source for course scheduling and delivery information.

Retaining duplicated metadata across both datasets would introduce unnecessary redundancy and increase the risk of categorical inconsistencies during downstream integration.

In [4]:
enrollment_cleaned = drop_columns(
    df=enrollment,
    cols_to_drop=ENROLLMENT_REDUNDANT_METADATA_COLS,
    verbose=True,
)

Columns succesfully dropped: ['shift', 'weekday', 'schedule_time', 'delivery_mode', 'campus']


## Normalize Operational Categorical Variables

Operational categorical variables exhibited inconsistent formatting, typographical variations, and fragmented semantic labels.

The normalization pipeline applies three sequential transformation stages:

1. textual normalization,
2. typographical correction,
3. and canonical taxonomy consolidation.

This process ensures consistent analytical representations while preserving deterministic and auditable transformation logic.

### Text Standardization

Basic textual normalization is applied to operational categorical fields in order to reduce purely formatting-related inconsistencies prior to semantic consolidation.

This stage includes operations such as whitespace trimming and lowercase normalization.

In [5]:
offering_normalized = normalize_text_columns(
    df=offering,
    text_cols=OFFERING_TEXT_COLS,
    normalize_text=normalize_text,
    verbose=True,
)

Text columns normalized: ['shift', 'weekday', 'schedule_time', 'delivery_mode']


In [6]:
unique_values_before = offering[OFFERING_TEXT_COLS].nunique()
unique_values_after = offering_normalized[OFFERING_TEXT_COLS].nunique()

print("Number of unique values in offering text columns before and after normalization:")
for col in OFFERING_TEXT_COLS:
    print(f"{col:<15}: {unique_values_before[col]:<3} ->  {unique_values_after[col]:<3}")

Number of unique values in offering text columns before and after normalization:
shift          : 6   ->  4  
weekday        : 14  ->  7  
schedule_time  : 38  ->  34 
delivery_mode  : 26  ->  17 


### Typographical Variant Correction

Typographical inconsistencies identified during the audit stage are corrected through explicit deterministic mappings before canonical category assignment.

In [7]:
offering_cleaned = correct_typo_variants(
    df=offering_normalized,
    mappings_dict=TYPO_MAPPINGS,
    unmapped="nan",
    verbose=True,
    normalize_func=None,
)

Column: 'shift'
Number of unique categories before typo correction: 4
Number of unique categories after typo correction: 3
['noche' 'manana' 'tarde']

Column: 'delivery_mode'
Number of unique categories before typo correction: 17
Number of unique categories after typo correction: 3
['presencial' nan 'virtual']

Column: 'weekday'
Number of unique categories before typo correction: 7
Number of unique categories after typo correction: 6
['martes' 'sabado' 'jueves' 'lunes' 'viernes' 'miercoles']



### Residual Delivery Mode Inspection

After deterministic normalization of pure onsite (`presencial`) and online (`virtual`) modality descriptors, a subset of operational labels remained unmapped.

These residual values were explicitly inspected before further consolidation.

In [8]:
offering_normalized[
    offering_cleaned["delivery_mode"].isna()
    ]["delivery_mode"].value_counts()

delivery_mode
4 hs presencial y 2 virtual              26
virtual (con encuentros presenciales)     5
4 presencial y 2 virtual                  5
presencial y 2 hs virtual                 4
4 hs presenciales y 2 virtuales           4
4 hs presencial 2 virtual                 3
4 presencial, 2 virtual                   3
2 presencial y 4 virtual                  2
4 presenciales y 2 virtuales              1
3 hs presencial 3 virtual                 1
2 hs presencial y 2 virtual               1
2 hs practicas, 4 presenciales            1
3 presencial y 3 virtual                  1
Name: count, dtype: int64

The remaining unmapped modality descriptors consistently represented mixed instructional schemes combining onsite and remote instructional components (e.g., combinations of presencial and virtual hours).

Given the semantic consistency of these residual categories, they were consolidated under the analytical label `hibrida`.

In [9]:
offering_cleaned["delivery_mode"] = offering_cleaned["delivery_mode"].fillna("hibrida")
offering_cleaned["delivery_mode"].unique()

array(['presencial', 'hibrida', 'virtual'], dtype=object)

### Canonical Category Consolidation

Operational category labels are consolidated into a reduced analytical taxonomy through explicit semantic mappings.

This transformation reduces categorical fragmentation and improves consistency across downstream analytical workflows.

In [10]:
offering_canonical = apply_canonical_taxonomy(
    df=offering_cleaned,
    mappings_dict=CANONICAL_MAPPINGS,
    verbose=True,
)

Column: 'shift'
Unique categories after applying canonical taxonomy: ['night' 'morning' 'afternoon']

Column: 'delivery_mode'
Unique categories after applying canonical taxonomy: ['on-site' 'hybrid' 'online']

Column: 'weekday'
Unique categories after applying canonical taxonomy: ['tuesday' 'saturday' 'thursday' 'monday' 'friday' 'wednesday']



## Standardize Numerical Types

Minor datatype inconsistencies identified during the audit stage were corrected prior to dataset integration.

The `workload` variable, originally represented as floating-point despite containing integer-valued observations, was converted to a nullable integer dtype in order to improve semantic consistency and downstream interpretability.

In [11]:
offering_canonical["workload"] = (
    offering_canonical["workload"]
    .astype("Int64")
)

## Reconcile Enrollment Metric Inconsistencies

A small number of records presented minor discrepancies between the reported total enrollment and the aggregate recomputed from component enrollment metrics.

Given the negligible magnitude of these inconsistencies and the additive consistency observed across the component variables, a deterministic reconciliation rule was applied using the recomputed total while preserving the original institutional values for traceability purposes.

In [12]:
enrollment_cleaned = reconcile_metric_totals(
    df=enrollment_cleaned,
    component_columns=ENROLLMENT_METRIC_COMPONENT_COLS,
    reported_column="total_enrollment",
    verbose=True,
    tolerance=2,
    overwrite=True,
)

Metric reconciliation completed for 'total_enrollment' with tolerance of 2.
Component columns: ['dropout_count', 'insufficient_count', 'free_status_count', 'promoted_completion_count', 'regular_completion_count']
Original reported values in 'total_enrollment' overwritten with reconciled values.


The original institutional values are preserved under
`reported_total_enrollment`, while `total_enrollment`
now contains the reconciled analytical metric used for
downstream analysis.

In [13]:
enrollment_cleaned.loc[
    enrollment_cleaned["total_enrollment_difference"] != 0,
    [
        "reported_total_enrollment",
        "computed_total_enrollment",
        "reconciled_total_enrollment",
        "total_enrollment_difference",
    ],
]

,reported_total_enrollment,computed_total_enrollment,reconciled_total_enrollment,total_enrollment_difference
35,19,18,18,1
101,51,49,49,2
111,58,57,57,1
145,57,56,56,1


Auxiliary reconciliation columns used for auditability and validation during preprocessing are removed prior to persisting the final analytical dataset.

In [14]:
# Build reconciliation flags and final reconciled difference column for traceability
enrollment_cleaned["is_reconciled"] = enrollment_cleaned["total_enrollment_difference"] != 0
enrollment_cleaned["reconciled_enrollment_diff"] = enrollment_cleaned["total_enrollment_difference"]


reconciliation_redundant_cols = [
    "reported_total_enrollment",
    "computed_total_enrollment",
    "reconciled_total_enrollment",
    "total_enrollment_difference" 
]

enrollment_cleaned.drop(
    columns=reconciliation_redundant_cols,
    inplace=True,
)

## Integrate Operational Metadata

Operational metadata from the `Offering` table is merged into the enrollment analytical base using course and section identifiers.

This integration consolidates scheduling, modality, and campus information into a unified analysis-ready dataset.

In [15]:
analytical_base = enrollment_cleaned.merge(offering_canonical, on=["course_code", "section"], how="left")
print(f"Shape of analytical base: {analytical_base.shape}")

Shape of analytical base: (303, 18)


### Handle Unmatched Operational Records

A small number of enrollment records could not be matched against the canonical operational metadata table (`Offering`) during integration.

Given the absence of reliable scheduling and modality metadata for these observations, the unmatched records were excluded from the final analytical dataset.

In [16]:
unmatched_mask = (
    analytical_base["delivery_mode"].isna()
)

print(
    f"Unmatched records: {unmatched_mask.sum()}"
)

Unmatched records: 6


In [17]:
enrollment.loc[
    unmatched_mask
].dropna(how='any').shape[0]

2

The inspection above shows that only a small subset of the unmatched observations retained partial operational metadata in the original enrollment source table.

Given the limited number of recoverable records, the incomplete nature of the residual metadata, and the additional pipeline complexity required to support secondary-source fallback integration, these observations were excluded in favor of preserving a simpler and fully deterministic preprocessing workflow.

In [18]:
analytical_base = analytical_base.loc[
    ~unmatched_mask
].copy()

print(f"Shape of analytical base: {analytical_base.shape}")

Shape of analytical base: (297, 18)


## Integrate Program Descriptors

Program-level descriptive information is incorporated through integration with the `Programs` table.

This enrichment step adds standardized program names to the analytical base while preserving the original program identifiers.

In [19]:
analytical_base = analytical_base.merge(programs, on="program_code", how="left")

## Validate Integrated Dataset

Post-integration validation checks are performed to verify structural consistency across the final analytical dataset.

The validation stage includes:

- row preservation checks,
- duplicate detection,
- null profile inspection,
- and verification of reconciliation outputs.

In [20]:
analytical_base.shape

(297, 19)

In [21]:
analytical_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   course_name                 297 non-null    object
 1   section                     297 non-null    int64 
 2   total_enrollment            297 non-null    int64 
 3   dropout_count               297 non-null    int64 
 4   insufficient_count          297 non-null    int64 
 5   free_status_count           297 non-null    int64 
 6   promoted_completion_count   297 non-null    int64 
 7   regular_completion_count    297 non-null    int64 
 8   course_code                 297 non-null    object
 9   program_code                297 non-null    object
 10  is_reconciled               297 non-null    bool  
 11  reconciled_enrollment_diff  297 non-null    int64 
 12  workload                    297 non-null    Int64 
 13  shift                       297 non-null    object

In [22]:
analytical_base.duplicated(
    subset=["course_code", "section"]
).sum()

np.int64(0)

In [23]:
analytical_base.isna().sum()

course_name                   0
section                       0
total_enrollment              0
dropout_count                 0
insufficient_count            0
free_status_count             0
promoted_completion_count     0
regular_completion_count      0
course_code                   0
program_code                  0
is_reconciled                 0
reconciled_enrollment_diff    0
workload                      0
shift                         0
weekday                       0
schedule_time                 0
delivery_mode                 0
campus                        0
program_name                  0
dtype: int64

## Persist Processed Analytical Dataset

The final cleaned analytical dataset is persisted for downstream exploratory analysis, visualization, and statistical evaluation workflows.

In [24]:
from src.utils.io import export_parquet

export_parquet(
    df=analytical_base,
    path="data/clean/analytical_base.parquet",
)

## Final Remarks

The deterministic cleaning and integration pipeline produced a structurally consistent analytical dataset with successful validation outcomes across merge integrity, duplicate detection, missingness inspection, and metric reconciliation checks.

At this stage, the analytical base is considered suitable for downstream exploratory and statistical analysis workflows.

Before proceeding to exploratory analysis, a lightweight feature engineering stage will be introduced in order to derive interpretable analytical indicators and ratio-based metrics intended to improve downstream analytical expressiveness while preserving the original institutional structure of the data.